# 02 - Demo inference CNN + OpenCV + Roboflow trên Colab

**Vai trò trong sản phẩm:** Chạy thử pipeline từ ảnh khay nguyên tấm: cắt ô bằng OpenCV, CNN phân loại món, Roboflow đếm trứng, xuất hóa đơn.

**Bản nộp đã được sắp xếp lại:** notebook này giữ nguyên luồng xử lý chính, chỉ đổi tên file/thêm mô tả để dễ đọc khi nộp.

Để an toàn khi nộp/chia sẻ, API key Roboflow không được hardcode trong bản này; khi chạy, Colab sẽ hỏi key hoặc đọc từ biến môi trường `ROBOFLOW_API_KEY`.


# Canteen Inference V2: CNN + OpenCV Crop + Roboflow Egg Counter

Notebook này chạy demo tính tiền từ **ảnh khay nguyên tấm**:

1. Resize ảnh khay về `800x600` để tọa độ OpenCV luôn khớp.
2. Cắt 5 ô trên khay.
3. **Resize từng ô crop về `224x224`** trước khi đưa vào CNN.
4. CNN nhận diện món.
5. Nếu món liên quan `thit_kho` / `thit_kho_trung`, gọi Roboflow để đếm `egg whole` và `egg half`.
6. Tính tiền, lưu bill JSON/CSV/TXT.

> Lưu ý quan trọng: model CNN train từ notebook `224` có lớp `Rescaling(1/255)` bên trong model, nên input inference **không chia `/255` lần nữa** nếu phát hiện model đã có lớp Rescaling.


In [ ]:
# Cell 1 - Cài thư viện cần thiết
!pip -q install inference-sdk


In [ ]:
# Cell 2 - Mount Google Drive và khai báo đường dẫn

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, math, shutil, time
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/canteen_project')

# ===== Model CNN =====
MODEL_DIR = ROOT / 'models' / 'cnn'
CNN_MODEL_PATH = MODEL_DIR / 'final_canteen_cnn.keras'
CLASS_NAMES_PATH = MODEL_DIR / 'class_names.json'
MENU_PATH = MODEL_DIR / 'menu.json'

# ===== Ảnh khay nguyên tấm =====
RAW_TRAYS_DIR = ROOT / 'data' / 'raw_trays'
TEST_TRAYS_DIR = ROOT / 'data' / 'test_trays'
EMPTY_TRAYS_DIR = ROOT / 'data' / 'empty_trays'

# ===== Output inference =====
INFER_DIR = ROOT / 'inference'
CROPPED_DIR = INFER_DIR / 'cropped_slots'              # crop đúng kích thước ô thực tế sau khi cắt
CROPPED_224_DIR = INFER_DIR / 'cropped_slots_224'      # crop đã resize 224x224 để CNN dùng
DEBUG_BOXES_DIR = INFER_DIR / 'debug_boxes'
EGG_DEBUG_DIR = INFER_DIR / 'egg_debug'
EGG_INPUT_DIR = EGG_DEBUG_DIR / 'input_crops'
EGG_ANNOTATED_DIR = EGG_DEBUG_DIR / 'annotated'
EGG_JSON_DIR = EGG_DEBUG_DIR / 'result_json'
PREDICTIONS_DIR = INFER_DIR / 'predictions'

BILLS_DIR = ROOT / 'outputs' / 'bills'
LOGS_DIR = ROOT / 'outputs' / 'logs'

for p in [MODEL_DIR, RAW_TRAYS_DIR, TEST_TRAYS_DIR, EMPTY_TRAYS_DIR,
          CROPPED_DIR, CROPPED_224_DIR, DEBUG_BOXES_DIR,
          EGG_DEBUG_DIR, EGG_INPUT_DIR, EGG_ANNOTATED_DIR, EGG_JSON_DIR,
          PREDICTIONS_DIR, BILLS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ===== Kích thước chuẩn =====
TRAY_SIZE = (800, 600)   # (width, height) cho ảnh khay nguyên tấm để crop tọa độ
CNN_IMG_SIZE = (224, 224)  # (width, height) cho từng ô đưa vào CNN

print('ROOT:', ROOT)
print('CNN_MODEL_PATH:', CNN_MODEL_PATH)
print('CLASS_NAMES_PATH:', CLASS_NAMES_PATH)
print('MENU_PATH:', MENU_PATH)


In [ ]:
# Cell 3 - Hàm phụ trợ

def list_files(folder, exts=('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
    folder = Path(folder)
    files = []
    for ext in exts:
        files.extend(folder.glob(f'*{ext}'))
        files.extend(folder.glob(f'*{ext.upper()}'))
    return sorted(files)


def show_bgr(img_bgr, title=None, figsize=(8, 6)):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()


def format_vnd(n):
    return f"{int(n):,}".replace(',', '.') + 'đ'

print('Ảnh test_trays:', len(list_files(TEST_TRAYS_DIR)))
for f in list_files(TEST_TRAYS_DIR)[:10]:
    print(' -', f.name)

print('\nẢnh raw_trays:', len(list_files(RAW_TRAYS_DIR)))
for f in list_files(RAW_TRAYS_DIR)[:10]:
    print(' -', f.name)

print('\nẢnh empty_trays:', len(list_files(EMPTY_TRAYS_DIR)))
for f in list_files(EMPTY_TRAYS_DIR)[:10]:
    print(' -', f.name)


## Cell 4 - Tọa độ cắt khay

Tọa độ này dùng cho ảnh khay đã resize về `800x600`.

Format mỗi box:

```python
[ymin, ymax, xmin, xmax]
```


In [ ]:
# Cell 4 - Tọa độ OpenCV cho ảnh khay 800x600

TARGET_W, TARGET_H = TRAY_SIZE

COMPARTMENTS = {
    'o_lon_tren_trai':  [45, 280, 55, 385],
    'o_lon_tren_phai':  [45, 280, 480, 750],
    'o_nho_duoi_trai':  [335, 550, 45, 270],
    'o_nho_duoi_giua':  [335, 550, 315, 495],
    'o_nho_duoi_phai':  [335, 550, 540, 760],
}

SLOT_LABELS_VI = {
    'o_lon_tren_trai': 'Ô lớn trên trái',
    'o_lon_tren_phai': 'Ô lớn trên phải',
    'o_nho_duoi_trai': 'Ô nhỏ dưới trái',
    'o_nho_duoi_giua': 'Ô nhỏ dưới giữa',
    'o_nho_duoi_phai': 'Ô nhỏ dưới phải',
}

COMPARTMENTS


In [ ]:
# Cell 5 - Cắt 5 ô từ ảnh khay nguyên tấm


def read_image_cv2(image_path):
    image_path = Path(image_path)
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f'Không đọc được ảnh: {image_path}')
    return img


def crop_slots_from_tray(image_path, save=True, pad=0):
    """
    Input: ảnh khay nguyên tấm.

    Luồng resize đúng:
      1) Resize ảnh khay nguyên tấm -> 800x600 để cắt bằng tọa độ cố định.
      2) Cắt từng ô.
      3) Resize từng ô -> 224x224 để đưa vào CNN.

    Output:
      - crops: dict slot_name -> crop BGR ở kích thước ô thật sau khi cắt
      - crops_224: dict slot_name -> crop BGR đã resize 224x224
      - debug_img: ảnh 800x600 có vẽ box
      - resized_img: ảnh khay 800x600 BGR
    """
    image_path = Path(image_path)
    img = read_image_cv2(image_path)
    resized_img = cv2.resize(img, (TARGET_W, TARGET_H))
    debug_img = resized_img.copy()

    crops = {}
    crops_224 = {}

    raw_out_dir = CROPPED_DIR / image_path.stem
    resized_out_dir = CROPPED_224_DIR / image_path.stem

    if save:
        raw_out_dir.mkdir(parents=True, exist_ok=True)
        resized_out_dir.mkdir(parents=True, exist_ok=True)

    for slot_name, box in COMPARTMENTS.items():
        ymin, ymax, xmin, xmax = box
        ymin2 = max(0, ymin + pad)
        ymax2 = min(TARGET_H, ymax - pad)
        xmin2 = max(0, xmin + pad)
        xmax2 = min(TARGET_W, xmax - pad)

        crop = resized_img[ymin2:ymax2, xmin2:xmax2]
        crop_224 = cv2.resize(crop, CNN_IMG_SIZE)

        crops[slot_name] = crop
        crops_224[slot_name] = crop_224

        if save:
            cv2.imwrite(str(raw_out_dir / f'{slot_name}.jpg'), crop)
            cv2.imwrite(str(resized_out_dir / f'{slot_name}_224.jpg'), crop_224)

        cv2.rectangle(debug_img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(debug_img, slot_name, (xmin, max(20, ymin - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

    if save:
        DEBUG_BOXES_DIR.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(DEBUG_BOXES_DIR / f'{image_path.stem}_debug.jpg'), debug_img)

    return crops, crops_224, debug_img, resized_img


In [ ]:
# Cell 6 - Test cắt thử 1 ảnh

candidate_images = list_files(TEST_TRAYS_DIR) or list_files(RAW_TRAYS_DIR) or list_files(EMPTY_TRAYS_DIR)

if not candidate_images:
    print('Chưa có ảnh nào. Hãy bỏ ảnh khay nguyên tấm vào data/test_trays/ hoặc data/raw_trays/.')
else:
    sample_path = candidate_images[0]
    print('Đang test crop ảnh:', sample_path)
    crops, crops_224, debug_img, resized_img = crop_slots_from_tray(sample_path, save=True, pad=0)

    show_bgr(debug_img, title=f'Debug boxes: {sample_path.name}', figsize=(10, 7))

    for slot, crop_224 in crops_224.items():
        show_bgr(crop_224, title=f'{slot} - crop 224x224 cho CNN', figsize=(3, 3))


In [ ]:
# Cell 7 - Load CNN model, class_names và menu giá

import tensorflow as tf
from tensorflow import keras

if not CNN_MODEL_PATH.exists():
    raise FileNotFoundError(f'Thiếu model CNN: {CNN_MODEL_PATH}')
if not CLASS_NAMES_PATH.exists():
    raise FileNotFoundError(f'Thiếu class_names.json: {CLASS_NAMES_PATH}')

model = keras.models.load_model(CNN_MODEL_PATH)

with open(CLASS_NAMES_PATH, 'r', encoding='utf-8') as f:
    class_names = json.load(f)

# Nếu class_names lưu dạng dict index->name thì đổi về list theo thứ tự index
if isinstance(class_names, dict):
    class_names = [class_names[str(i)] for i in range(len(class_names))]

# Menu mặc định theo đề. Nếu có menu.json thì ưu tiên menu.json.
DEFAULT_MENU = {
    'com': 10000,
    'com_trang': 10000,
    'dau_hu_sot_ca': 25000,
    'ca_hu_kho': 30000,
    'thit_kho_trung': 30000,
    'thit_kho': 25000,
    'canh_chua_co_ca': 25000,
    'canh_chua_khong_ca': 10000,
    'suon_nuong': 30000,
    'canh_rau': 7000,
    'rau_xao': 10000,
    'trung_chien': 25000,
    'khay_trong': 0,
    'empty': 0,
    'background': 0,
}

if MENU_PATH.exists():
    with open(MENU_PATH, 'r', encoding='utf-8') as f:
        menu = json.load(f)
else:
    menu = DEFAULT_MENU.copy()
    with open(MENU_PATH, 'w', encoding='utf-8') as f:
        json.dump(menu, f, ensure_ascii=False, indent=2)
    print('Chưa có menu.json nên đã tạo menu mặc định:', MENU_PATH)

# Kiểm tra model có lớp Rescaling chưa.
# Model train của bạn có layers.Rescaling(1/255), nên inference KHÔNG được /255 thêm lần nữa.
def has_rescaling_layer(m):
    for layer in m.layers:
        if layer.__class__.__name__.lower() == 'rescaling':
            return True
        if 'rescaling' in layer.name.lower():
            return True
        # Nếu data_augmentation/Sequential lồng trong model
        if hasattr(layer, 'layers'):
            for sub in layer.layers:
                if sub.__class__.__name__.lower() == 'rescaling' or 'rescaling' in sub.name.lower():
                    return True
    return False

MODEL_HAS_RESCALING = has_rescaling_layer(model)

print('Số class:', len(class_names))
print('class_names:', class_names)
print('\nModel có Rescaling bên trong không?:', MODEL_HAS_RESCALING)
print('\nMenu:')
print(json.dumps(menu, ensure_ascii=False, indent=2))


In [ ]:
# Cell 8 - Hàm predict một crop bằng CNN

CONF_THRESHOLD = 0.45


def normalize_label(label):
    """Chuẩn hóa tên class để map giá dễ hơn."""
    s = str(label).strip().lower()
    s = s.replace(' ', '_').replace('-', '_')
    while '__' in s:
        s = s.replace('__', '_')
    return s


def prepare_crop_for_cnn(crop_bgr):
    """
    Input crop BGR bất kỳ kích thước nào.
    Output batch tensor shape (1, 224, 224, 3).

    Lưu ý:
    - Nếu model đã có Rescaling(1/255), đưa ảnh 0..255 vào.
    - Nếu model không có Rescaling, tự chia /255.
    """
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop_rgb = cv2.resize(crop_rgb, CNN_IMG_SIZE)
    arr = crop_rgb.astype('float32')

    if not MODEL_HAS_RESCALING:
        arr = arr / 255.0

    arr = np.expand_dims(arr, axis=0)
    return arr


def predict_crop_cnn(crop_bgr, top_k=3):
    x = prepare_crop_for_cnn(crop_bgr)
    probs = model.predict(x, verbose=0)[0]
    top_idx = probs.argsort()[-top_k:][::-1]

    top_preds = []
    for idx in top_idx:
        raw_label = class_names[int(idx)]
        top_preds.append({
            'class': normalize_label(raw_label),
            'raw_class': raw_label,
            'confidence': float(probs[int(idx)])
        })

    best = top_preds[0]
    return best, top_preds

# Test nhanh nếu đã crop ở cell trước
if 'crops' in globals() and crops:
    for slot, crop in crops.items():
        pred, top_preds = predict_crop_cnn(crop)
        print(slot, '=>', pred, '| top3:', top_preds)


## Cell 9 - Roboflow egg detector

Model Roboflow của bạn có 2 class:

- `egg whole` = 1 quả trứng
- `egg half` = 0.5 quả trứng

Bản nộp không hardcode API key. Khi chạy demo, notebook sẽ đọc biến môi trường `ROBOFLOW_API_KEY` hoặc yêu cầu nhập key bằng `getpass`.


In [ ]:
# Cell 9 - Khởi tạo Roboflow client bằng API key nhập khi chạy

import os
from getpass import getpass
from inference_sdk import InferenceHTTPClient

ROBOFLOW_MODEL_ID = 'egg2-3omrr/1'
ROBOFLOW_API_URL = 'https://serverless.roboflow.com'

# Không hardcode API key trong file nộp. Colab sẽ đọc biến môi trường hoặc hỏi key.
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '').strip()
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass('Nhập Roboflow API key: ').strip()

if not ROBOFLOW_API_KEY:
    raise ValueError('Thiếu ROBOFLOW_API_KEY nên không thể gọi Roboflow.')

rf_client = InferenceHTTPClient(
    api_url=ROBOFLOW_API_URL,
    api_key=ROBOFLOW_API_KEY
)

print('Đã khởi tạo Roboflow client cho model:', ROBOFLOW_MODEL_ID)


In [ ]:
# Cell 10 - Hàm đếm egg whole / egg half bằng Roboflow

EGG_CONF_THRESHOLD = 0.35


def parse_egg_class(cls_name):
    s = str(cls_name).strip().lower().replace('_', ' ').replace('-', ' ')
    s = ' '.join(s.split())
    if s in ['egg whole', 'whole egg', 'eggwhole']:
        return 'egg whole'
    if s in ['egg half', 'half egg', 'egghalf']:
        return 'egg half'
    return s


def draw_egg_predictions(crop_bgr, predictions, out_path):
    img = crop_bgr.copy()
    for p in predictions:
        conf = float(p.get('confidence', 0))
        if conf < EGG_CONF_THRESHOLD:
            continue

        cls = parse_egg_class(p.get('class', ''))
        x = float(p.get('x', 0))
        y = float(p.get('y', 0))
        w = float(p.get('width', 0))
        h = float(p.get('height', 0))

        x1 = int(x - w / 2)
        y1 = int(y - h / 2)
        x2 = int(x + w / 2)
        y2 = int(y + h / 2)

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(img, f'{cls} {conf:.2f}', (x1, max(18, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

    cv2.imwrite(str(out_path), img)
    return img


def count_eggs_roboflow(crop_bgr, crop_name='crop.jpg', save_debug=True):
    """
    Trả về dict:
      egg_whole: số object class egg whole
      egg_half: số object class egg half
      egg_equiv: egg_whole + egg_half * 0.5
      billable_eggs: ceil(egg_equiv)
    """
    EGG_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    EGG_ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)
    EGG_JSON_DIR.mkdir(parents=True, exist_ok=True)

    crop_path = EGG_INPUT_DIR / crop_name
    cv2.imwrite(str(crop_path), crop_bgr)

    result = rf_client.infer(str(crop_path), model_id=ROBOFLOW_MODEL_ID)
    predictions = result.get('predictions', []) if isinstance(result, dict) else []

    egg_whole = 0
    egg_half = 0
    kept_predictions = []

    for p in predictions:
        conf = float(p.get('confidence', 0))
        if conf < EGG_CONF_THRESHOLD:
            continue

        cls = parse_egg_class(p.get('class', ''))
        if cls == 'egg whole':
            egg_whole += 1
        elif cls == 'egg half':
            egg_half += 1
        else:
            continue

        p2 = dict(p)
        p2['class_normalized'] = cls
        kept_predictions.append(p2)

    egg_equiv = egg_whole + egg_half * 0.5
    billable_eggs = int(math.ceil(egg_equiv)) if egg_equiv > 0 else 0

    egg_info = {
        'egg_whole': int(egg_whole),
        'egg_half': int(egg_half),
        'egg_equiv': float(egg_equiv),
        'billable_eggs': int(billable_eggs),
        'predictions': kept_predictions,
        'confidence_threshold': EGG_CONF_THRESHOLD,
        'skipped': False,
    }

    if save_debug:
        json_path = EGG_JSON_DIR / (Path(crop_name).stem + '_egg.json')
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(egg_info, f, ensure_ascii=False, indent=2)

        ann_path = EGG_ANNOTATED_DIR / (Path(crop_name).stem + '_egg_annotated.jpg')
        draw_egg_predictions(crop_bgr, kept_predictions, ann_path)

    return egg_info


In [ ]:
# Cell 11 - Logic tính tiền từng món

THIT_KHO_TRUNG_BASE_PRICE = 30000
EXTRA_EGG_PRICE = 6000

# Nếu tên class CNN khác tên trong menu thì map ở đây
CLASS_ALIASES = {
    'cơm_trắng': 'com',
    'com_trang': 'com',
    'com': 'com',
    'dau_hu_sot_ca': 'dau_hu_sot_ca',
    'dau_hu': 'dau_hu_sot_ca',
    'ca_hu_kho': 'ca_hu_kho',
    'thit_kho_trung': 'thit_kho_trung',
    'thit_kho': 'thit_kho',
    'canh_chua_co_ca': 'canh_chua_co_ca',
    'canh_chua_khong_ca': 'canh_chua_khong_ca',
    'suon_nuong': 'suon_nuong',
    'canh_rau': 'canh_rau',
    'rau_xao': 'rau_xao',
    'trung_chien': 'trung_chien',
    'khay_trong': 'khay_trong',
    'empty': 'khay_trong',
    'background': 'khay_trong',
}

EGG_CANDIDATE_CLASSES = {'thit_kho', 'thit_kho_trung'}


def canonical_class(pred_class):
    pred_class = normalize_label(pred_class)
    return CLASS_ALIASES.get(pred_class, pred_class)


def is_candidate_for_egg_detection(pred_class, top_preds=None):
    pred_class = canonical_class(pred_class)
    if pred_class in EGG_CANDIDATE_CLASSES:
        return True

    # Nếu top3 có thit_kho hoặc thit_kho_trung thì vẫn thử Roboflow.
    if top_preds:
        for p in top_preds:
            c = canonical_class(p.get('class', ''))
            if c in EGG_CANDIDATE_CLASSES:
                return True
    return False


def price_for_prediction(pred_class, egg_info=None):
    """
    Quy tắc:
    - thit_kho = 25k, không có trứng.
    - thit_kho_trung = 30k gồm 1 trứng.
    - Thêm 1 trứng + 6k.
    - egg whole = 1, egg half = 0.5, khi tính tiền dùng ceil.
    - Nếu CNN đoán thit_kho nhưng Roboflow thấy trứng, đổi thành thit_kho_trung.
    """
    pred_class = canonical_class(pred_class)
    egg_info = egg_info or {}

    egg_equiv = float(egg_info.get('egg_equiv', 0.0) or 0.0)
    billable_eggs = int(egg_info.get('billable_eggs', 0) or 0)

    final_class = pred_class
    note = ''

    if pred_class == 'thit_kho' and egg_equiv > 0:
        final_class = 'thit_kho_trung'
        note = 'CNN đoán thit_kho nhưng Roboflow phát hiện trứng nên đổi thành thit_kho_trung.'

    if final_class == 'thit_kho_trung':
        # Giá gốc đã gồm 1 trứng. Nếu không detect được trứng, vẫn tính 1 phần theo class CNN.
        charged_eggs = max(1, billable_eggs)
        extra_eggs = max(0, charged_eggs - 1)
        price = THIT_KHO_TRUNG_BASE_PRICE + extra_eggs * EXTRA_EGG_PRICE

        if note:
            note += ' '
        note += f'Trứng quy đổi: {egg_equiv:g}, tính tiền: {charged_eggs} trứng, thêm: {extra_eggs}.'
        return final_class, int(price), note

    price = int(menu.get(final_class, DEFAULT_MENU.get(final_class, 0)))
    return final_class, price, note


In [ ]:
# Cell 12 - Chạy inference cho một ảnh khay: crop -> resize 224 -> CNN -> Roboflow egg -> bill


def infer_one_tray(image_path, save=True, show=False):
    image_path = Path(image_path)
    crops, crops_224, debug_img, resized_img = crop_slots_from_tray(image_path, save=save, pad=0)

    items = []
    total = 0

    for slot_name, crop in crops.items():
        # predict_crop_cnn tự resize crop về 224x224 và xử lý đúng Rescaling
        pred, top_preds = predict_crop_cnn(crop, top_k=3)
        pred_class = canonical_class(pred['class'])

        if is_candidate_for_egg_detection(pred_class, top_preds):
            crop_name = f'{image_path.stem}_{slot_name}.jpg'
            egg_info = count_eggs_roboflow(crop, crop_name=crop_name, save_debug=True)
        else:
            egg_info = {
                'egg_whole': 0,
                'egg_half': 0,
                'egg_equiv': 0.0,
                'billable_eggs': 0,
                'predictions': [],
                'skipped': True,
                'reason': 'Not egg candidate'
            }

        final_class, price, note = price_for_prediction(pred_class, egg_info=egg_info)
        total += price

        item = {
            'slot': slot_name,
            'slot_vi': SLOT_LABELS_VI.get(slot_name, slot_name),
            'pred_class': pred_class,
            'raw_pred_class': pred['raw_class'],
            'confidence': float(pred['confidence']),
            'top3': top_preds,
            'final_class': final_class,
            'price': int(price),
            'price_vnd': format_vnd(price),
            'egg_info': egg_info,
            'note': note,
        }
        items.append(item)

    bill = {
        'image_name': image_path.name,
        'image_path': str(image_path),
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'items': items,
        'total': int(total),
        'total_vnd': format_vnd(total),
    }

    if save:
        bill_json = BILLS_DIR / f'{image_path.stem}_bill.json'
        with open(bill_json, 'w', encoding='utf-8') as f:
            json.dump(bill, f, ensure_ascii=False, indent=2)

        bill_txt = BILLS_DIR / f'{image_path.stem}_bill.txt'
        with open(bill_txt, 'w', encoding='utf-8') as f:
            f.write(f'HÓA ĐƠN: {image_path.name}\n')
            f.write('=' * 50 + '\n')
            for item in items:
                f.write(f"{item['slot_vi']}: {item['final_class']} - {item['price_vnd']} - conf={item['confidence']:.3f}\n")
                if item['egg_info'].get('egg_equiv', 0) > 0:
                    f.write(f"  egg whole={item['egg_info']['egg_whole']}, egg half={item['egg_info']['egg_half']}, quy đổi={item['egg_info']['egg_equiv']}\n")
                if item['note']:
                    f.write(f"  Ghi chú: {item['note']}\n")
            f.write('=' * 50 + '\n')
            f.write(f"TỔNG: {bill['total_vnd']}\n")

    print('\n' + '=' * 70)
    print('Ảnh:', image_path.name)
    print('=' * 70)
    for item in items:
        print(f"{item['slot_vi']:<18} | {item['final_class']:<22} | {item['price_vnd']:<10} | conf={item['confidence']:.3f}")
        if item['egg_info'].get('egg_equiv', 0) > 0:
            print(f"  ↳ egg whole={item['egg_info']['egg_whole']}, egg half={item['egg_info']['egg_half']}, quy đổi={item['egg_info']['egg_equiv']}")
        if item['note']:
            print('  ↳', item['note'])
    print('-' * 70)
    print('TỔNG:', bill['total_vnd'])

    if show:
        show_bgr(debug_img, title=f'Debug boxes: {image_path.name}', figsize=(10, 7))

    return bill


In [ ]:
# Cell 13 - Demo chạy 1 ảnh đầu tiên trong data/test_trays/
# Nếu test_trays rỗng, notebook sẽ lấy ảnh đầu tiên trong raw_trays/.

images = list_files(TEST_TRAYS_DIR) or list_files(RAW_TRAYS_DIR)

if not images:
    print('Chưa có ảnh khay để inference. Hãy bỏ ảnh vào data/test_trays/ hoặc data/raw_trays/.')
else:
    image_path = images[0]
    bill = infer_one_tray(image_path, save=True, show=True)


In [ ]:
# Cell 14 - Chạy hàng loạt tất cả ảnh trong data/test_trays/
# Nếu muốn chạy raw_trays thì đổi INPUT_DIR = RAW_TRAYS_DIR.

INPUT_DIR = TEST_TRAYS_DIR
# INPUT_DIR = RAW_TRAYS_DIR

all_images = list_files(INPUT_DIR)
all_bills = []

for image_path in all_images:
    print('\nĐang xử lý:', image_path.name)
    bill = infer_one_tray(image_path, save=True, show=False)
    all_bills.append(bill)

summary_rows = []
for bill in all_bills:
    row = {
        'image_name': bill['image_name'],
        'total': bill['total'],
        'total_vnd': bill['total_vnd'],
    }
    for item in bill['items']:
        row[item['slot']] = item['final_class']
        row[item['slot'] + '_price'] = item['price']
        row[item['slot'] + '_conf'] = item['confidence']
        row[item['slot'] + '_egg_equiv'] = item['egg_info'].get('egg_equiv', 0)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_csv = LOGS_DIR / 'inference_summary.csv'
summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')

print('\nĐã lưu summary:', summary_csv)
display(summary_df)


## Ghi chú sửa lỗi nhanh

### 1. Crop lệch ô
Sửa tọa độ trong `COMPARTMENTS`. Nhớ ảnh khay luôn được resize về `800x600` trước khi cắt.

### 2. CNN đoán sai bất thường
Kiểm tra `MODEL_HAS_RESCALING`:

- Nếu `True`: notebook đưa input 0..255 vào model, vì model tự chia `/255`.
- Nếu `False`: notebook tự chia `/255` trước khi predict.

Với notebook train CNN 224 từ scratch của bạn, model có `layers.Rescaling(1.0 / 255)`, nên **không chia `/255` ngoài model**.

### 3. API Roboflow
API key không hardcode trong bản nộp. Khi chạy Cell 9, nhập key Roboflow nếu Colab hỏi.

### 4. Tính trứng
Notebook đang dùng:

- `egg whole` = 1
- `egg half` = 0.5
- số trứng tính tiền = `ceil(egg_whole + egg_half * 0.5)`

Món `thit_kho_trung` có giá gốc 30.000đ gồm 1 trứng; từ trứng thứ 2 trở đi cộng thêm 6.000đ/trứng.
